# **Projet 3 - Prédiction de la consommation énergétique des bâtiments de Seattle**

 * Prédire la consommation énergétique des bâtiments non résidentiels de Seattle à partir de leurs caractéristiques structurelles.
 * Réaliser une analyse exploratoire afin d'identifier les principales tendances et anomalies des données.
 * Comparer plusieurs modèles de régression supervisée pour sélectionner le plus performant.
 * Identifier les variables ayant le plus d'influence sur la consommation énergétique.*
 

## **Objectf** 
Prédire la consommation énergétique (ou les émissions) des bâtiments non résidentiels afin d'aider la ville de Seattle à atteindre la neutralité carbone en 2050.

## **ÉTAPE 2** 

### **Réalisez votre feature engeneering**

#### **Importation des Modules**

In [3]:
#importation des librairies

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Importation des librairies OK")

Importation des librairies OK


In [4]:
#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor


print("Importation des modules OK")

Importation des modules OK


#### **Chargement du jeu de données**

In [5]:
df = pd.read_csv("../data/processed/buildings_clean.csv")

print("Importation du dataset terminée.")
print(f"Dimensions du dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Importation du dataset terminée.
Dimensions du dataset : 3348 lignes, 47 colonnes


#### **Exploration du jeu de données**

Avant de créer de nouvelles variables, il est important de vérifier la structure du jeu de données importé.

Cette première inspection permet de :
- vérifier les dimensions du dataset ;
- identifier les variables disponibles ;
- contrôler les types de données ;
- préparer les prochaines étapes du feature engineering.

In [6]:
# Aperçu général du dataset

display(df.head())

print(f"Nombre de lignes : {df.shape[0]}")
print(f"Nombre de colonnes : {df.shape[1]}")

,osebuildingid,datayear,buildingtype,primarypropertytype,propertyname,address,city,state,zipcode,taxparcelidentificationnumber,...,electricity(kbtu),naturalgas(therms),naturalgas(kbtu),defaultdata,comments,compliancestatus,outlier,totalghgemissions,ghgemissionsintensity,usage_type
0,1,2016,NonResidential,Hotel,Mayflower park hotel,405 Olive way,Seattle,WA,98101.0,0659000030,...,3946027.0,12764.52930,1276453.0,False,NaN,Compliant,NaN,249.98,2.83,Mono-usage
1,2,2016,NonResidential,Hotel,Paramount Hotel,724 Pine street,Seattle,WA,98101.0,0659000220,...,3242851.0,51450.81641,5145082.0,False,NaN,Compliant,NaN,295.86,2.86,Multi-usages
2,3,2016,NonResidential,Hotel,5673-The Westin Seattle,1900 5th Avenue,Seattle,WA,98101.0,0659000475,...,49526664.0,14938.00000,1493800.0,False,NaN,Compliant,NaN,2089.28,2.19,Mono-usage
3,5,2016,NonResidential,Hotel,HOTEL MAX,620 STEWART ST,Seattle,WA,98101.0,0659000640,...,2768924.0,18112.13086,1811213.0,False,NaN,Compliant,NaN,286.43,4.67,Mono-usage
4,8,2016,NonResidential,Hotel,WARWICK SEATTLE HOTEL (ID8),401 LENORA ST,Seattle,WA,98121.0,0659000970,...,5368607.0,88039.98438,8803998.0,False,NaN,Compliant,NaN,505.01,2.88,Multi-usages


Nombre de lignes : 3348
Nombre de colonnes : 47


In [7]:
# Informations générales

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3348 entries, 0 to 3347
Data columns (total 47 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   osebuildingid                    3348 non-null   int64  
 1   datayear                         3348 non-null   int64  
 2   buildingtype                     3348 non-null   str    
 3   primarypropertytype              3348 non-null   str    
 4   propertyname                     3348 non-null   str    
 5   address                          3348 non-null   str    
 6   city                             3348 non-null   str    
 7   state                            3348 non-null   str    
 8   zipcode                          3332 non-null   float64
 9   taxparcelidentificationnumber    3348 non-null   str    
 10  councildistrictcode              3348 non-null   int64  
 11  neighborhood                     3348 non-null   str    
 12  latitude                       

#### **Identification des variables disponibles**
identifier les différentes features disponibles dans le jeu de données.

In [8]:
# Liste des colonnes

pd.set_option("display.max_rows", None)

display(pd.DataFrame({
    "Feature": df.columns,
    "Type": df.dtypes.values
}))

,Feature,Type
0,osebuildingid,int64
1,datayear,int64
2,buildingtype,str
3,primarypropertytype,str
4,propertyname,str
5,address,str
6,city,str
7,state,str
8,zipcode,float64
9,taxparcelidentificationnumber,str


> - Les variables directement liées à la consommation énergétique seront exclues afin d'éviter tout phénomène de **data leakage**.
> - Les nouvelles features seront principalement issues :
>  des caractéristiques structurelles du bâtiment ;
>  de son ancienneté ;
>  de son usage ;
>  de sa localisation.

### **Feature Engineering**

#### **Création de la variable BuildingAge**


> - Transformation de l'année de construction en âge du bâtiment.
> - Cette variable permet de mieux représenter l'ancienneté du bâtiment.
> - Elle est créée sans risque de **data leakage**.

In [10]:
# Création de l'âge du bâtiment

df["BuildingAge"] = df["datayear"] - df["yearbuilt"]

# Vérification
df[["yearbuilt", "datayear", "BuildingAge"]].head()

,yearbuilt,datayear,BuildingAge
0,1927,2016,89
1,1996,2016,20
2,1969,2016,47
3,1926,2016,90
4,1980,2016,36


#### **Création de la variable ParkingRatio**

> - Calcul de la proportion de la surface dédiée au parking.
> - Cette variable décrit la structure du bâtiment.
> - Elle est créée sans risque de **data leakage**.

In [11]:
# Création du ratio de surface de parking

df["ParkingRatio"] = (
    df["propertygfaparking"] / df["propertygfatotal"]
)

# Vérification
df[["propertygfaparking", "propertygfatotal", "ParkingRatio"]].head()

,propertygfaparking,propertygfatotal,ParkingRatio
0,0,88434,0.000000
1,15064,103566,0.145453
2,196718,956110,0.205748
3,0,61320,0.000000
4,62000,175580,0.353115


#### **Création de la variable BuildingRatio**

> - Calcul de la proportion de la surface occupée par le bâtiment.
> - Cette variable complète la description de la structure du bâtiment.
> - Elle est créée sans risque de **data leakage**.

In [12]:
# Création du ratio de surface du bâtiment

df["BuildingRatio"] = (
    df["propertygfabuilding(s)"] / df["propertygfatotal"]
)

# Vérification
df[["propertygfabuilding(s)", "propertygfatotal", "BuildingRatio"]].head()

,propertygfabuilding(s),propertygfatotal,BuildingRatio
0,88434,88434,1.000000
1,88502,103566,0.854547
2,759392,956110,0.794252
3,61320,61320,1.000000
4,113580,175580,0.646885


#### **Création de la variable IsMultiUse**

> - Identification des bâtiments ayant plusieurs types d'usages.
> - Cette variable décrit la diversité des usages du bâtiment.
> - Elle est créée sans risque de **data leakage**.

In [13]:
# Création d'une variable binaire indiquant plusieurs usages

df["IsMultiUse"] = (
    df["listofallpropertyusetypes"]
    .str.contains(",", na=False)
    .astype(int)
)

# Vérification
df[["listofallpropertyusetypes", "IsMultiUse"]].head()

,listofallpropertyusetypes,IsMultiUse
0,Hotel,0
1,"Hotel, Parking, Restaurant",1
2,Hotel,0
3,Hotel,0
4,"Hotel, Parking, Swimming Pool",1


#### **Création de la variable NumberPropertyUses**

> - Comptage du nombre de types d'usages du bâtiment.
> - Cette variable complète l'information apportée par **IsMultiUse**.
> - Elle est créée sans risque de **data leakage**.

In [14]:
# Comptage du nombre de types d'usages

df["NumberPropertyUses"] = (
    df["listofallpropertyusetypes"]
    .str.split(",")
    .str.len()
)

# Vérification
df[["listofallpropertyusetypes", "NumberPropertyUses"]].head()

,listofallpropertyusetypes,NumberPropertyUses
0,Hotel,1
1,"Hotel, Parking, Restaurant",3
2,Hotel,1
3,Hotel,1
4,"Hotel, Parking, Swimming Pool",3


### **Conclusion**

> - Cinq nouvelles variables ont été créées à partir des informations existantes.
> - Les nouvelles variables couvrent différentes catégories (temporalité, structure et usage).
> - Le jeu de données est enrichi et prêt pour l'étape suivante de préparation des features.

### **Conclusion Feature Engineering**

> - Cinq nouvelles variables ont été créées afin d'enrichir le jeu de données.
> - Les variables peu pertinentes et celles susceptibles d'introduire du **data leakage** ont été supprimées.
> - Les variables explicatives (**X**) et la variable cible (**y**) ont été séparées.
> - Les variables catégorielles ont été préparées pour être encodées avec **OneHotEncoder** avant l'entraînement des modèles.
> - Le jeu de données est désormais prêt pour la phase de modélisation.

### **Sauvegarde du jeu de données**

In [15]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)

df.to_csv(
    "../data/processed/buildings_features.csv",
    index=False
)

print("Jeu de données enrichi sauvegardé avec succès.")

Jeu de données enrichi sauvegardé avec succès.
